# ⚙️ Fase 2 — Preprocessing & Feature Engineering
### Public Transport Delays · Omar Mora Flores

Limpieza, feature engineering, encoding y split. **Se excluye `delayed`** (es el target
binarizado → *data leakage*) y las columnas de alta cardinalidad (estaciones, IDs, fechas).

In [1]:
import numpy as np, pandas as pd, pickle
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
while not (ROOT / "data" / "public_transport_delays.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
TARGET = "actual_arrival_delay_min"; RS = 42
df = pd.read_csv(ROOT/"data"/"public_transport_delays.csv")
df["event_type"] = df["event_type"].fillna("None")
print("Shape:", df.shape)

Shape: (2000, 24)


## 2.1 Limpieza + Feature Engineering

In [2]:
# Features de ingeniería
df["has_event"] = (df["event_type"] != "None").astype(int)
df["extreme_weather"] = df["weather_condition"].isin(["Storm","Snow","Fog"]).astype(int)

drop = ["trip_id","date","time","route_id","origin_station","destination_station",
        "scheduled_departure","scheduled_arrival","delayed"]   # delayed = leakage
y = df[TARGET]
X = df.drop(columns=[c for c in drop if c in df] + [TARGET])
cat = X.select_dtypes("object").columns.tolist()
print("Categóricas a codificar:", cat)
print("Features de ingeniería: has_event, extreme_weather")

Categóricas a codificar: ['transport_type', 'weather_condition', 'event_type', 'season']
Features de ingeniería: has_event, extreme_weather


## 2.2 Encoding + split + escalado (fit en train)

In [3]:
X = pd.get_dummies(X, columns=cat, drop_first=True)
X = X.astype({c:"int" for c in X.select_dtypes("bool").columns})
FEATURES = X.columns.tolist()
print("Features tras encoding:", len(FEATURES))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RS)
cols_scale = ["temperature_C","humidity_percent","wind_speed_kmh","precipitation_mm",
              "event_attendance_est","traffic_congestion_index","actual_departure_delay_min"]
cols_scale = [c for c in cols_scale if c in X.columns]
scaler = StandardScaler()
X_tr=X_tr.copy(); X_te=X_te.copy()
X_tr[cols_scale] = scaler.fit_transform(X_tr[cols_scale])
X_te[cols_scale] = scaler.transform(X_te[cols_scale])
print(f"Train {X_tr.shape} | Test {X_te.shape}")

Features tras encoding: 28
Train (1600, 28) | Test (400, 28)


## 2.3 Guardar splits

In [4]:
pickle.dump({"X_train":X_tr,"X_test":X_te,"y_train":y_tr,"y_test":y_te,
             "features":FEATURES,"scaler":scaler,"cols_scaled":cols_scale},
            open(ROOT/"data"/"splits.pkl","wb"))
print("Guardado data/splits.pkl ·  ➡️ Siguiente: 03_modeling.ipynb")

Guardado data/splits.pkl ·  ➡️ Siguiente: 03_modeling.ipynb
